# Converting Emulators to Pyomo Expressions

This tutorial's purpose is to walk you through the process of converting your `AutoEmulate` Emulators to Pyomo algebraic expressions for easy optimization workflows. Currently only `PolynomialRegression` and `MLP` are supported for this conversion.

We'll demonstrate the following steps:
1. Training an MLP Emulator on a simple toy function using `AutoEmulate`
2. Exporting the trained Emulator into algebraic expressions compatible with Pyomo
3. Validating the Pyomo expressions against Emulator prediction

In [ ]:
# General imports for the notebook
import warnings
import numpy as np
import torch
warnings.filterwarnings("ignore")
np.random.seed(42)
torch.manual_seed(42)

## Toy simulation

Before we train an emulator with AutoEmulate, we need to get a set of input/output pairs from our simulation to use as training data.

Below is a simple toy simulation that computes the product of two inputs: `y = x1 * x2`.

In [ ]:
# Generate data: y = x1 * x2
n_samples = 1000
x1 = np.random.uniform(-100, 100, size=n_samples)
x2 = np.random.uniform(-100, 100, size=n_samples)

def F(x1, x2):
    return x1 * x2

x = np.column_stack((x1, x2))
y = F(x1, x2)

# Convert to tensors for AutoEmulate
x = torch.tensor(x, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32)

x.shape, y.shape

## Train Emulator

For this tutorial, we'll focus on training an MLP (Multi-Layer Perceptron) Emulator.

In [ ]:
from autoemulate import AutoEmulate

# Initialize AutoEmulate with MLP Emulator only
ae = AutoEmulate(x, y, log_level="info", models=['MLP'])
Emulator = ae.best_result()

## Converting to Pyomo

Now that we have a trained emulator, we can convert it to Pyomo algebraic expressions. This allows us to use the emulator in optimization problems with mathematical programming solvers.

Before conversion, we will setup our optimization framework through pyomo. Pyomo has a **ConcreteModel**, which is a container that holds all the components of an optimization problem - variables, objectives, constraints. Think of it as the "problem definition sheet."

- **Pyomo variables:** are symbolic placeholders for inputs that a mathematical solver can manipulate during optimization. They correspond to the same x1, x2 input features the Emulator was trained on. There are multiple types of variable domains in pyomo, some common are explained below:

    | Domain                 | Use case                                      |
    |------------------------|-----------------------------------------------|
    | `pyo.Reals`            | Continuous variables (default for regression) |
    | `pyo.NonNegativeReals` | Continuous, must be ≥ 0                       |
    | `pyo.NonPositiveReals` | Continuous, must be ≤ 0                       |
    | `pyo.Integers`         | Integer variables                             |
    | `pyo.Binary`           | 0/1 decisions                                 |
    | `pyo.PositiveReals`    | Strictly > 0                                  |
    | `pyo.NegativeReals`    | Strictly < 0                                  |


- **Pyomo algebraic expressions:** are the Emulator's learned input→output relationships written as pure math, which any mathematical programming solver can evaluate and search for an optimal solution.

In [ ]:
import pyomo.environ as pyo

# Create a Pyomo concrete model
pyo_model = pyo.ConcreteModel()

# Define decision variables (input variables for the autoemulate emulator)
# `pyo.Reals` are continuous real number. Because our training data is continuous, we use `pyo.Reals` for the decision variables. 
# If the training data were integers, we would use `pyo.Integers` instead. Bounds are set to (-100, 100) based on the range of the training data.
pyo_model.x1 = pyo.Var(domain=pyo.Reals, bounds=(-100, 100))
pyo_model.x2 = pyo.Var(domain=pyo.Reals, bounds=(-100, 100))

### Export Emulator to Pyomo

This is the core integration step. The `pyomofy` function converts the Emulator's weights, biases, and activation functions into explicit algebraic expressions.

It automatically handles:

* **Input Standardization:** Considering any scaling applied to inputs of the emulator during training.
* **Forward Pass:** inputs propagates through the architecture of the emulator and a output value is computed.
* **Output Inverse Scaling:** Inverting any scaling applied to the output of the Emulator.


* **ReLU Approximation:** If `nn.ReLU` was used during training of MLP Emulator, it's automatically approximated with Softplus (controlled by `relu_beta`, default `50`). if results mismatch significantly, increase `relu_beta` value (max 100) or use a smooth activation function like `SiLU` during training:

    ```
    from torch import nn

    # Train with SiLU activation for better Pyomo conversion
    ae = AutoEmulate(
        x, y, 
        models=['MLP'],
        model_params={'activation_cls': nn.SiLU}  # Use SiLU instead of ReLU
    )
    ```

In [ ]:
from autoemulate.convert.pyomo import pyomofy

# Convert the Emulator to Pyomo expressions
# Pass the Result object (or TransformedEmulator) and the list of Pyomo variables
# Returns a list of expressions (one per output of the Emulator.)
emulator_expressions = pyomofy(Emulator, [pyo_model.x1, pyo_model.x2], relu_beta=100)

# Since we only have 1 output for our Emulator, we take the first element of expression list emulator_expressions[0].
# For multi-output emulators, to access nth output, use emulator_expressions[n-1]
emulator_expr = emulator_expressions[0]

## Verify Numerical Equivalence

Before using the Pyomo expressions in optimization, we **must** verify that they yield the same values as the Emulator. This validation ensures the conversion was successful and the algebraic expressions accurately represent the Emulator.

In [ ]:
# Pick an input point to compare pyomo prediction with the Emulator's prediction. We picked first sample x[0] of the simulation data.

x_init = x[0]

# Set pyomo variables to the selected input values. item() convert tensors to python float for inputting to pyomo.
pyo_model.x1.set_value(x_init[0].item())
pyo_model.x2.set_value(x_init[1].item())

# 1. Predict Pyomo at picked input point
pyomo_val = pyo.value(emulator_expr)

# 2. Predict Emulator at the same input point

Emulator_input = x[0].reshape(1, -1)
Emulator_val = Emulator.model.predict(Emulator_input).item()

# Print comparison
print(f"Input point:        x1={x_init[0]:.4f}, x2={x_init[1]:.4f}")
print(f"Emulator prediction: {Emulator_val:.12f}")
print(f"Pyomo prediction:   {pyomo_val:.12f}")
print(f"\nPyomo vs Emulator:   {abs(pyomo_val - Emulator_val):.12f}")

## Using the Pyomo Expression

Now that we've validated the conversion, the Pyomo expression can be used in optimization problems. Here's a simple example of how you might set up an objective function:

In [ ]:
# Maximize the emulator output value.
# `sense=pyo.maximize` tells the solver to find inputs that make the expression as large as possible.
# `sense=pyo.minimize` tells the solver to find inputs that make the expression as small as possible.

pyo_model.obj = pyo.Objective(expr=emulator_expr, sense=pyo.maximize)

# Optionally, we can add constraints on the emulator output:
# pyo_model.constraint1 = pyo.Constraint(expr=emulator_expr >= 1000)

# OR restrict inputs to the some other data range:

# pyo_model.constraint2 = pyo.Constraint(expr=pyo.inequality(-50, pyo_model.x1, 100))
# pyo_model.constraint3 = pyo.Constraint(expr=pyo.inequality(-100, pyo_model.x2, 200))

# Or bounded on one side:

# pyo_model.constraint4 = pyo.Constraint(expr=pyo_model.x1 >= -100)
# pyo_model.constraint5 = pyo.Constraint(expr=pyo_model.x1 <= 100)

In [ ]:
# Solve the optimization problem using mathematical solvers.
# In this example, we will use IPOPT (a gradient-based NLP solver).
# Other options: 'glpk' (linear/integer), 'gurobi', 'bonmin' (mixed-integer nonlinear).

solver = pyo.SolverFactory('ipopt')
results = solver.solve(pyo_model, tee=True)  # tee=True prints solver log

# Extract and print results
print(f"Solver status:    {results.solver.status}")
print(f"Objective value:  {pyo.value(pyo_model.obj):.6f}")
print(f"Optimal x1:       {pyo.value(pyo_model.x1):.6f}")
print(f"Optimal x2:       {pyo.value(pyo_model.x2):.6f}")